# Construyendo Aplicaciones de Generación de Imágenes (OpenAI)

Hay más en los LLM que la generación de texto. También puedes generar imágenes a partir de descripciones de texto. Las imágenes como modalidad son útiles en MedTech, arquitectura, turismo, desarrollo de juegos, marketing y más. En esta lección trabajamos con los modelos **GPT Image** actuales en la plataforma OpenAI.

## Objetivos de aprendizaje

Al final de esta lección podrás:

- Explicar qué es la generación de imágenes y dónde es útil.
- Entender la familia de modelos `gpt-image` y cómo difiere de los modelos heredados DALL·E.
- Construir una aplicación de generación de imágenes y editar imágenes.

## ¿Qué es la generación de imágenes?

Los modelos de generación de imágenes crean imágenes a partir de un mensaje de texto. Los modelos modernos como `gpt-image` aprenden la relación entre texto e imágenes durante el entrenamiento, y luego transforman iterativamente ruido aleatorio en una imagen que coincide con tu mensaje.

Dos familias conocidas de modelos de imagen son:

- **`gpt-image` (OpenAI)** — la generación actual usada en esta lección. Soporta generación de imagen a partir de texto y edición de imágenes (re-pintado con una máscara).
- **Midjourney** — un modelo popular de terceros con su propio servicio y flujo de trabajo basado en Discord.

> Los modelos de imagen más antiguos de OpenAI — **DALL·E 2** y **DALL·E 3** — son heredados. DALL·E 3 ya no está disponible para nuevas implementaciones, y características como `create_variation` solo existían en DALL·E 2. Usa los modelos `gpt-image` para nuevas aplicaciones.

> **Importante:** los modelos `gpt-image` devuelven la imagen generada como **base64** (`b64_json`), no como una URL. Tu código decodifica la cadena base64 a bytes y la guarda — no hay una URL de imagen para descargar.


## Construyendo tu primera aplicación de generación de imágenes

¿Entonces, qué se necesita para construir una aplicación de generación de imágenes? Necesitas las siguientes bibliotecas:

- **python-dotenv**, se recomienda encarecidamente usar esta biblioteca para mantener tus secretos en un archivo *.env* separado del código.
- **openai**, esta biblioteca es la que usarás para interactuar con la API de OpenAI.
- **pillow**, para trabajar con imágenes en Python.


1. Crea un archivo *.env* con el siguiente contenido:

    ```text
    OPENAI_API_KEY='<add your OpenAI key here>'
    ```



1. Recoja las bibliotecas anteriores en un archivo llamado *requirements.txt* de la siguiente manera:

    ```text
    python-dotenv
    openai
    pillow
    ```


1. A continuación, cree un entorno virtual e instale las bibliotecas:


In [ ]:
# create virtual env
! python3 -m venv venv
# activate environment
! source venv/bin/activate
# install libraries
# pip install -r requirements.txt, if using a requirements.txt file 
! pip install python-dotenv openai pillow

> [!NOTE]
> Para Windows, usa los siguientes comandos para crear y activar tu entorno virtual:

    ```bash
    python3 -m venv venv
    venv\Scripts\activate.bat
    ````

1. Añade el siguiente código en un archivo llamado *app.py*:

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai

    # importar dotenv
    dotenv.load_dotenv()

    # Crear objeto OpenAI (lee OPENAI_API_KEY de tu .env)
    client = openai.OpenAI()


    try:
        # Crear una imagen usando la API de generación de imágenes
        generation_response = client.images.generate(
            model="gpt-image-1",
            prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Ingresa tu texto de indicación aquí
            size='1024x1024',
            n=1
        )
        # Establecer el directorio para la imagen almacenada
        image_dir = os.path.join(os.curdir, 'images')

        # Si el directorio no existe, crearlo
        if not os.path.isdir(image_dir):
            os.mkdir(image_dir)

        # Inicializar la ruta de la imagen (nota que el tipo de archivo debe ser png)
        image_path = os.path.join(image_dir, 'generated-image.png')

        # los modelos gpt-image devuelven la imagen en base64 (b64_json), no una URL
        image_b64 = generation_response.data[0].b64_json
        generated_image = base64.b64decode(image_b64)
        with open(image_path, "wb") as image_file:
            image_file.write(generated_image)

        # Mostrar la imagen en el visor de imágenes predeterminado
        image = Image.open(image_path)
        image.show()

    # capturar excepciones
    except openai.BadRequestError as err:
        print(err)

    ```

Vamos a explicar este código:

- Primero, importamos las librerías que necesitamos, incluyendo la librería OpenAI, la librería dotenv, el módulo base64 y la librería Pillow.

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai
    ```

- Después, creamos el cliente, que lee la clave API desde tu archivo ``.env``.

    ```python
    # Crear objeto OpenAI
    client = openai.OpenAI()
    ```

- A continuación, generamos la imagen:

    ```python
    generation_response = client.images.generate(
        model="gpt-image-1",
        prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Introduzca el texto de su solicitud aquí
        size='1024x1024',
        n=1
    )
    ```

    Los modelos `gpt-image` devuelven la imagen como una cadena **base64** en `data[0].b64_json`. La decodificamos a bytes y la escribimos en un archivo — no hay URL para descargar.

    ```python
    image_b64 = generation_response.data[0].b64_json
    generated_image = base64.b64decode(image_b64)
    with open(image_path, "wb") as image_file:
        image_file.write(generated_image)
    ```

- Por último, abrimos la imagen y usamos el visor de imágenes estándar para mostrarla:

    ```python
    image = Image.open(image_path)
    image.show()
    ```

### Más detalles sobre la generación de imágenes

Veamos los parámetros de `images.generate`:

- **model**, es el modelo de imagen a usar (por ejemplo `gpt-image-1`).
- **prompt**, es el texto que se usa para generar la imagen. Aquí es "Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils".
- **size**, es el tamaño de la imagen generada (por ejemplo `1024x1024`, `1536x1024`, `1024x1536` o `"auto"`).
- **n**, es el número de imágenes generadas. Aquí generamos una.

> Los modelos de imagen no toman un parámetro `temperature` — eso es un control para generación de texto. Para obtener variedad, llama a la API otra vez; para reducir la variedad, haz tu prompt más específico.

## Capacidades adicionales de la generación de imágenes

Has visto cómo generar una imagen con unas pocas líneas de Python. Los modelos `gpt-image` también pueden **editar** una imagen existente. Al proporcionar una imagen, una **máscara** opcional (que marca el área a cambiar) y un prompt, puedes alterar parte de una imagen — por ejemplo, ponerle un sombrero a nuestro conejo.

```python
response = client.images.edit(
  model="gpt-image-1",
  image=open("base_image.png", "rb"),
  mask=open("mask.png", "rb"),
  prompt="An image of a rabbit with a hat on its head.",
  n=1,
  size="1024x1024"
)
# las ediciones también se devuelven en base64
edited_image = base64.b64decode(response.data[0].b64_json)
```

La imagen base contiene solo el conejo; la imagen final tiene el sombrero en el conejo.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Descargo de responsabilidad**:
Este documento ha sido traducido utilizando el servicio de traducción automática [Co-op Translator](https://github.com/Azure/co-op-translator). Aunque nos esforzamos por la precisión, tenga en cuenta que las traducciones automatizadas pueden contener errores o inexactitudes. El documento original en su idioma nativo debe considerarse la fuente autorizada. Para información crítica, se recomienda una traducción profesional humana. No somos responsables de cualquier malentendido o interpretación errónea que surja del uso de esta traducción.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
